In [ ]:
!pip install anyio fastapi httpx2 matplotlib numpy uvicorn

In [ ]:
import time
from socket import socket

import httpx2
import matplotlib.pyplot as plt
import numpy as np
from anyio import create_task_group, open_process

In [ ]:
async def make_request(client, data, i):
    t0 = time.monotonic()
    await client.get(url)
    t1 = time.monotonic()
    data[i] = t1 - t0

In [ ]:
async def make_requests(data, concurrently):
    t0 = time.monotonic()
    async with httpx2.AsyncClient() as client:
        async with create_task_group() as tg:
            for i in range(data.size):
                if concurrently:
                    tg.start_soon(make_request, client, data, i)
                else:
                    await make_request(client, data, i)
    t1 = time.monotonic()
    return t1 - t0

In [ ]:
with socket() as sock:
    sock.bind(("127.0.0.1", 0))
    port = sock.getsockname()[1]

url = f"http://127.0.0.1:{port}"
n = 1_000
sequential = np.empty(n)
parallel = np.empty(n)

async with await open_process(["uvicorn", "server:app", "--port", str(port)]) as process:
    while True:
        try:
            httpx2.get(url)
        except Exception:
            time.sleep(0.1)
        else:
            break
    t_sequential = await make_requests(sequential, concurrently=False)
    t_parallel = await make_requests(parallel, concurrently=True)
    process.terminate()
    await process.wait()

In [ ]:
plt.plot(sequential, label=f"sequential ({round(t_sequential, 1)} s)")
plt.plot(parallel, label=f"parallel ({round(t_parallel, 1)} s)")
plt.xlabel("requests")
plt.ylabel("seconds")
plt.legend()
plt.show()